<a href="https://colab.research.google.com/github/dtc-oss/python_hw/blob/hw_7/hw7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Парсинг рубрики «Технологии» на Lifehacker

Задача — с помощью `requests` и `BeautifulSoup` собрать заголовки и тексты материалов с первых десяти страниц рубрики.

Порядок работы: получить страницы рубрики, собрать ссылки на статьи, открыть каждую статью, извлечь заголовок и основной текст и объединить результаты в `DataFrame`.

## 1. Импорт библиотек и настройки

### Формат ссылки для пагинации.
В адресе рубрики номер страницы передаётся через параметр `page`: например, `?page=2`.

https://lifehacker.ru/topics/technology/?page=2

Для первой страницы параметр не нужен. Между HTTP-запросами предусмотрена небольшая пауза.

In [1]:
import json
import time
from urllib.parse import urljoin, urlparse, urlunparse, urldefrag

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

CATEGORY_URL = "https://lifehacker.ru/topics/technology/"
BASE_DOMAIN = "lifehacker.ru"
PAGES_TO_PARSE = 10
REQUEST_DELAY = 0.4  # пауза между запросами

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/130.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.8",
}

session = requests.Session()
session.headers.update(HEADERS)


def get_soup(url):
    """Загружает страницу и возвращает объект BeautifulSoup."""
    response = session.get(url, timeout=25)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    time.sleep(REQUEST_DELAY)
    return soup


## 3. Функции для получения страниц и ссылок

Ссылки нормализуются и проверяются: оставляем внутренние ссылки на отдельные материалы, исключая рубрики и служебные страницы.

In [2]:
def normalize_url(url):
    """Убирает параметры, якоря и лишние варианты записи URL."""
    url, _ = urldefrag(url)
    parsed = urlparse(url)
    host = parsed.netloc.lower()
    if host.startswith("www."):
        host = host[4:]
    path = parsed.path.rstrip("/") + "/"
    return urlunparse(("https", host, path, "", "", ""))


def is_article_url(url):
    """Оставляет ссылки на отдельные статьи, а не на рубрики и служебные страницы."""
    parsed = urlparse(url)
    host = parsed.netloc.lower()
    if host.startswith("www."):
        host = host[4:]
    if host != BASE_DOMAIN:
        return False

    path = parsed.path.strip("/")
    if not path:
        return False

    # У статей Lifehacker обычно один сегмент пути: /nazvanie-stati/
    parts = path.split("/")
    if len(parts) != 1:
        return False

    slug = parts[0].lower()
    excluded = {
        "topics", "topic", "tag", "author", "authors", "search",
        "about", "faq", "contacts", "wp-json", "feed", "comments",
        "page", "category", "privacy-policy", "terms",
    }
    if slug in excluded or "." in slug:
        return False

    return True


def get_article_links(soup, page_url):
    """Находит ссылки на статьи в списке материалов рубрики."""
    root = soup.find("main") or soup

    # Сначала ищем ссылки, которые находятся в заголовках карточек.
    title_links = []
    for link in root.find_all("a", href=True):
        if link.find(["h2", "h3", "h4"]) or link.find_parent(["h2", "h3", "h4"]):
            title_links.append(link)

    # На случай, если заголовки карточек размечены не тегами h2-h4.
    links_to_check = title_links if title_links else root.find_all("a", href=True)

    result = []
    for link in links_to_check:
        link_text = " ".join(link.stripped_strings)
        if not link_text:
            continue

        absolute_url = normalize_url(urljoin(page_url, link["href"]))
        if is_article_url(absolute_url):
            result.append(absolute_url)

    # Удаляем дубликаты, сохраняя первоначальный порядок.
    return list(dict.fromkeys(result))


## 4. Извлечение заголовка и текста статьи

Заголовок ищется в `h1`. Для содержимого используется набор возможных CSS-селекторов. Если текст не найден в HTML-контейнере, код дополнительно проверяет структурированные данные `JSON-LD`.

In [3]:
def get_jsonld_article_body(soup):
    """Ищет текст статьи в структурированных данных JSON-LD, если он там есть."""
    def find_body(value):
        if isinstance(value, dict):
            body = value.get("articleBody")
            if isinstance(body, str) and body.strip():
                return body
            for nested_value in value.values():
                found = find_body(nested_value)
                if found:
                    return found
        elif isinstance(value, list):
            for nested_value in value:
                found = find_body(nested_value)
                if found:
                    return found
        return None

    for script in soup.select('script[type="application/ld+json"]'):
        try:
            data = json.loads(script.get_text())
        except (json.JSONDecodeError, TypeError):
            continue
        body = find_body(data)
        if body:
            return BeautifulSoup(body, "html.parser").get_text("\n", strip=True)
    return ""


def clean_element_text(element):
    """Убирает меню и вспомогательные элементы, оставляя текст основной статьи."""
    for unwanted in element.select(
        "script, style, noscript, button, form, nav, aside, "
        ".share, .social-share, .comments, .related-posts, .advertisement"
    ):
        unwanted.decompose()

    lines = [line.strip() for line in element.get_text("\n").splitlines() if line.strip()]
    return "\n".join(lines)


def get_article_text(soup):
    """Извлекает основной текст материала, используя несколько возможных селекторов."""
    content_selectors = [
        '[itemprop="articleBody"]',
        ".article__content",
        ".article__body",
        ".article__text",
        ".article-content",
        ".entry-content",
        ".post-content",
        ".content__article",
        "main article",
        "article",
        "main",
    ]

    for selector in content_selectors:
        element = soup.select_one(selector)
        if element:
            text = clean_element_text(element)
            # Пустые и совсем короткие блоки обычно являются не текстом статьи,
            # а заголовком или служебным элементом страницы.
            if len(text) > 80:
                return text

    # Резервный способ - текст articleBody из JSON-LD.
    return get_jsonld_article_body(soup)


def parse_article(url):
    """Получает заголовок и текст одной статьи."""
    soup = get_soup(url)

    title_element = soup.select_one("main h1") or soup.find("h1")
    title = title_element.get_text(" ", strip=True) if title_element else ""

    if not title:
        og_title = soup.find("meta", property="og:title")
        if og_title:
            title = og_title.get("content", "").strip()

    text = get_article_text(soup)
    return {"title": title, "text": text, "url": url}


## 5. Собираем ссылки с первых десяти страниц рубрики

In [4]:
article_urls = []
failed_list_pages = []

for page in tqdm(range(1, PAGES_TO_PARSE + 1), desc="Страницы рубрики"):
    if page == 1:
        page_url = CATEGORY_URL
    else:
        page_url = f"{CATEGORY_URL}?page={page}"

    try:
        page_soup = get_soup(page_url)
        page_links = get_article_links(page_soup, page_url)
        article_urls.extend(page_links)
        print(f"Страница {page}: найдено ссылок — {len(page_links)}")
    except requests.RequestException as error:
        failed_list_pages.append(page_url)
        print(f"Не удалось загрузить {page_url}: {error}")

# Удаляем статьи, которые встретились на нескольких страницах.
article_urls = list(dict.fromkeys(article_urls))
print(f"Всего уникальных ссылок на статьи: {len(article_urls)}")

Страницы рубрики:   0%|          | 0/10 [00:00<?, ?it/s]

Страница 1: найдено ссылок — 44
Страница 2: найдено ссылок — 45
Страница 3: найдено ссылок — 45
Страница 4: найдено ссылок — 44
Страница 5: найдено ссылок — 45
Страница 6: найдено ссылок — 44
Страница 7: найдено ссылок — 45
Страница 8: найдено ссылок — 45
Страница 9: найдено ссылок — 45
Страница 10: найдено ссылок — 45
Всего уникальных ссылок на статьи: 312


## 6. Обходим ссылки и извлекаем заголовки и тексты

In [5]:
articles = []
failed_articles = []

for url in tqdm(article_urls, desc="Статьи"):
    try:
        article = parse_article(url)
        if article["title"] and article["text"]:
            articles.append(article)
        else:
            failed_articles.append(url)
    except requests.RequestException as error:
        failed_articles.append(url)
        print(f"Не удалось загрузить статью {url}: {error}")

print(f"Успешно обработано статей: {len(articles)}")
print(f"Не удалось обработать ссылок: {len(failed_articles)}")

Статьи:   0%|          | 0/312 [00:00<?, ?it/s]

Успешно обработано статей: 312
Не удалось обработать ссылок: 0


## 7. Создаём датафрейм и сохраняем результат

In [6]:
pd.set_option("display.max_colwidth", 300)

data = pd.DataFrame(articles, columns=["title", "text", "url"])
data = data.drop_duplicates(subset="url").reset_index(drop=True)
display(data.head())

data.to_csv("lifehacker_technology_10_pages.csv", index=False, encoding="utf-8-sig")
print("Результат сохранён в lifehacker_technology_10_pages.csv")

,title,text,url
0,Время есть — лучшие рецепты Лайфхакера,"30 мин.\nБыстрые белковые пирожки из теста фило — можно и в духовке, и на сковороде, и в аэрогриле\nПоделиться\n2 ч.\nВлажный апельсиновый кекс без кефира и молока\nПоделиться\n1 ч.\nРецепт сочных слоек с клюквой из готового слоёного теста\nПоделиться\n30 мин.\nПростые куриные оладушки с луком и...",https://lifehacker.ru/recipes/
1,"Лайфхакер Здоровье: рассказываем, как принимать решения о здоровье, опираясь на принципы доказательной медицины","Здоровье\nПринимайте решения о своём здоровье на основе достоверной информации\nМы проверили все факты и отвечаем за свои слова!\n100+\nврачей-рецензентов\n5 000+\nстатей о здоровье\nО разделе\nкак мы работаем\n12 идей, как расслабиться без алкоголя\n0\n16 октября\nЗОЖ\nБолезни\nПрепараты\nПерва...",https://lifehacker.ru/health/
2,Новый мини-фен Xiaomi способен сушить короткие волосы за минуту,"Xiaomi\nанонсировала\nвысокоскоростной фен Mijia High-Speed Hair Dryer Mini весом всего 270 граммов. Новинка на 30% меньше и на 20% легче модели\nH501\n, выпущенной в 2023 году, но сохранила скорость вращения мотора на уровне 110 000 оборотов в минуту.\n⚡ Новости из мира гаджетов, обзоры, гайды ...",https://lifehacker.ru/predstavlen-novyi-fen-xiaomi/
3,Оптимус Прайм для тяжёлых работ: в Китае показали огромного и мощного робота Neo1,"Китайская компания NeodraDynamics Robotics\nпоказала\nчетырёхметрового промышленного робота Neo1. Гигант в стиле «Трансформеров» способен поднимать около 500 килограммов груза благодаря сочетанию гидравлических и электрических приводов.\n⚡ Новости из мира гаджетов, обзоры, гайды и не только — в ...",https://lifehacker.ru/predstavlen-ogromnyi-robot-neo1/
4,Объясняем за минуту: как увеличить шрифт на телефоне,"Мелкий текст на экране быстро утомляет глаза, а людям постарше иногда и вовсе мешает пользоваться смартфоном. Увеличить его можно в настройках, никаких сторонних приложений не потребуется. Вот как это сделать.\n⚡ Новости из мира гаджетов, обзоры, гайды и не только — в телеграм-канале «\nТехнолог...",https://lifehacker.ru/kak-uvelichit-shrift-na-telefone/


Результат сохранён в lifehacker_technology_10_pages.csv
